# Tutorial 02: Adding the Router (Protecting the Vector Store)

In Tutorial 01, we saw that sending a simple "Hello" to a naive RAG system caused it to perform an unnecessary vector search, waste compute resources, and hallucinate an answer based on irrelevant documents.

In a production system, every vector search and LLM generation costs money and adds latency. We need to protect our expensive retrieval operations from trivial or irrelevant queries.

In this notebook, we will introduce the **Single Responsibility Principle (SRP)** by building a dedicated Router. The Router's only job is to analyze the user's intent and decide where the query should go *before* we touch the database.

### Prerequisites
* Completion of Tutorial 01.
* A local running instance of Ollama.


### Step 1: Defining the Routes

Instead of a single, linear path, our system will now have two distinct paths:

1. **`general_chat`**: For greetings, casual conversation, and off-topic questions. Bypasses the vector store entirely.
2. **`vector_search`**: For specific questions requiring domain knowledge. Routes to the FAISS/Qdrant database.

Let's build a simple LLM-based router using LangChain. We will force the LLM to output a structured JSON response containing only the route name.


In [ ]:
import json
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# 1. Initialize the LLM (using a fast, lightweight model like Llama 3)
# We set temperature to 0 because routing must be highly deterministic.
llm = Ollama(model="llama3", temperature=0)

# 2. Define the routing prompt
router_template = """You are a highly efficient semantic routing system. 
Your job is to analyze the user's query and route it to the correct subsystem.

Available Routes:
- "general_chat": Use this for greetings (e.g., "hello", "hi"), casual conversation, or questions that do not require specialized knowledge.
- "vector_search": Use this for technical questions, specific factual inquiries, or queries requiring internal documentation.

Respond ONLY with a valid JSON object containing a single key 'route'. Do not add any conversational text.

Query: {question}
JSON Output:"""

router_prompt = PromptTemplate.from_template(router_template)

# 3. Build the Router Chain
router_chain = router_prompt | llm

print("✅ Semantic Router initialized.")


### Step 2: Testing the Semantic Router

Let's test our new router with the exact same queries that broke our naive system in Tutorial 01.


In [ ]:
def execute_route(query: str):
    """Simulates the routing logic."""
    print(f"User Query: '{query}'")
    
    # 1. Invoke the router
    raw_response = router_chain.invoke({"question": query})
    
    try:
        # Parse the JSON response
        decision = json.loads(raw_response)
        route = decision.get("route")
        print(f"🔀 Router Decision: Routed to [{route}]")
        
        # 2. Execute path based on the route
        if route == "general_chat":
            print("🤖 System Action: Bypassing vector store. Generating generic conversational response...")
        elif route == "vector_search":
            print("📚 System Action: Triggering Vector Database Search...")
        else:
            print("⚠️ System Action: Unknown route. Falling back to vector search.")
            
    except json.JSONDecodeError:
         print(f"⚠️ Error parsing router output: {raw_response}")

print("--- Test 1: The Chatty User ---")
execute_route("Hello, good morning! How are you?")

print("\n--- Test 2: The Technical User ---")
execute_route("What hardware is generative inference fenced to?")


### The Architectural Win

By adding these 30 lines of code, you have just eliminated a massive percentage of unnecessary vector database load:

* **Latency:** A user saying "Hello" gets a response in milliseconds rather than waiting for an embedding and retrieval cycle.
* **Cost/Compute:** Your embedding models and CPU threads are freed up to handle actual technical queries.
* **Reliability:** The LLM is no longer forced to hallucinate connections between casual greetings and technical documents.

*Note: In the production codebase of this repository, you will find this logic heavily optimized with a 0ms heuristic regex fast-path inside [`scripts/router.py`](../scripts/router.py).*

---

### 🏋️ Challenge

The current router is binary. In large enterprise systems, you might have multiple vector databases (e.g., one for HR policies, one for Engineering docs).

**Your Task:**
Update the `router_template` to support three routes: `general_chat`, `hr_database`, and `engineering_database`. Test it by feeding it the query: *"What is the PTO policy for new hires?"* Ensure it routes successfully without defaulting to the engineering tools.

In **Tutorial 03: Building the Grader**, we will handle the next big failure mode: what happens when the Router correctly chooses `vector_search`, but the Vector Database returns garbage documents?
